In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import tqdm
import os
import wandb


In [2]:
# Hyperparameters
mb_size = 64
Z_dim = 1000
h_dim = 128
lr = 1e-3
y_dim=10

In [3]:
# Load MNIST data
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1))  # Flatten the 28x28 image to 784
])

train_dataset = datasets.MNIST(root='../MNIST', train=True, transform=transform, download=True)
train_loader = DataLoader(train_dataset, batch_size=mb_size, shuffle=True)

X_dim = 784  # 28 x 28

In [4]:
# Xavier Initialization
def xavier_init(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [5]:
class Generator(nn.Module):
    def __init__(self, z_dim, y_dim, h_dim, x_dim):
        super().__init__()
        self.fc1 = nn.Linear(z_dim + y_dim, h_dim)
        self.fc2 = nn.Linear(h_dim, x_dim)

    def forward(self, z, y):
        x = torch.cat([z, y], dim=1)  #we concatenate the noise and the label
        h = F.relu(self.fc1(x))
        out = torch.sigmoid(self.fc2(h))
        return out

In [6]:
class Discriminator(nn.Module):
    def __init__(self, x_dim, y_dim, h_dim):
        super().__init__()
        self.fc1 = nn.Linear(x_dim + y_dim, h_dim)
        self.fc2 = nn.Linear(h_dim, 1)

    def forward(self, x, y):
        x = torch.cat([x, y], dim=1)  #we concatenate the label
        h = F.relu(self.fc1(x))
        out = self.fc2(h)        
        return out

In [7]:
def cGANTraining(G, D, loss_fn, train_loader):

    G.train()
    D.train()

    D_loss_real_total = 0
    D_loss_fake_total = 0
    G_loss_total = 0

    t = tqdm.tqdm(train_loader)

    for it, (X_real, labels) in enumerate(t):

        # ======================
        # DATA PREP
        # ======================
        X_real = X_real.float().to(device)

        labels = labels.to(device)
        y = F.one_hot(labels, num_classes=10).float().to(device)

        batch_size = X_real.size(0)

        ones_label = torch.ones(batch_size, 1).to(device)
        zeros_label = torch.zeros(batch_size, 1).to(device)

        # ======================
        # NOISE
        # ======================
        z = torch.randn(batch_size, Z_dim).to(device)

        # ======================
        # DISCRIMINATOR
        # ======================
        G_sample = G(z, y)

        D_real = D(X_real, y)
        D_fake = D(G_sample.detach(), y)

        D_loss_real = loss_fn(D_real, ones_label)
        D_loss_fake = loss_fn(D_fake, zeros_label)

        D_loss = D_loss_real + D_loss_fake

        D_solver.zero_grad()
        D_loss.backward()
        D_solver.step()

        # ======================
        # GENERATOR
        # ======================
        z = torch.randn(batch_size, Z_dim).to(device)
        G_sample = G(z, y)

        D_fake = D(G_sample, y)

        G_loss = loss_fn(D_fake, ones_label)

        G_solver.zero_grad()
        G_loss.backward()
        G_solver.step()

        # ======================
        # LOGGING FIX
        # ======================
        D_loss_real_total += D_loss_real.item()
        D_loss_fake_total += D_loss_fake.item()
        G_loss_total += G_loss.item()

    # ======================
    # AVERAGE LOSSES
    # ======================
    D_loss_real_avg = D_loss_real_total / len(train_loader)
    D_loss_fake_avg = D_loss_fake_total / len(train_loader)
    D_loss_avg = D_loss_real_avg + D_loss_fake_avg
    G_loss_avg = G_loss_total / len(train_loader)

    wandb.log({
        "D_loss_real": D_loss_real_avg,
        "D_loss_fake": D_loss_fake_avg,
        "D_loss": D_loss_avg,
        "G_loss": G_loss_avg
    })

    return G, D, G_loss_avg, D_loss_avg

In [8]:
def save_sample(G, epoch, mb_size, Z_dim, digit=0):

    out_dir = "out_cgan"
    G.eval()

    with torch.no_grad():

        z = torch.randn(mb_size, Z_dim).to(device)

        labels = torch.full((mb_size,), digit).to(device)
        y = F.one_hot(labels, num_classes=10).float().to(device)

        samples = G(z, y).detach().cpu().numpy()[:16]

    fig = plt.figure(figsize=(4, 4))
    gs = gridspec.GridSpec(4, 4)

    for i, sample in enumerate(samples):
        ax = plt.subplot(gs[i])
        plt.axis('off')
        ax.imshow(sample.reshape(28, 28), cmap='gray')

    if not os.path.exists(out_dir):
        os.makedirs(out_dir)

    plt.savefig(f"{out_dir}/{str(epoch).zfill(3)}.png", bbox_inches='tight')
    plt.close(fig)

In [9]:
wandb_log = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

G = Generator(Z_dim, y_dim, h_dim, X_dim).to(device)
D = Discriminator(X_dim, y_dim, h_dim).to(device)

G_solver = optim.Adam(G.parameters(), lr=lr)
D_solver = optim.Adam(D.parameters(), lr=lr)

loss_fn = nn.BCEWithLogitsLoss()   # 🔥 FIX IMPORTANT

if wandb_log:
    wandb.init(project="Lab2_task3")

    wandb.config.update({
        "y_dim": y_dim,
        "batch_size": mb_size,
        "Z_dim": Z_dim,
        "X_dim": X_dim,
        "h_dim": h_dim,
        "lr": lr,
    })

best_g_loss = float('inf')
save_dir = 'checkpoints'
os.makedirs(save_dir, exist_ok=True)

epochs = 100

for epoch in range(epochs):

    G, D, G_loss_avg, D_loss_avg = cGANTraining(
        G, D, loss_fn, train_loader
    )

    print(f"epoch {epoch}; D_loss: {D_loss_avg:.4f}; G_loss: {G_loss_avg:.4f}")

    if G_loss_avg < best_g_loss:
        best_g_loss = G_loss_avg
        torch.save(G.state_dict(), os.path.join(save_dir, 'G_best.pth'))
        torch.save(D.state_dict(), os.path.join(save_dir, 'D_best.pth'))

    save_sample(G, epoch, mb_size, Z_dim, digit=3)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: linneahejsupergroup (linneahejsupergroup-lule-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


100%|██████████| 938/938 [00:10<00:00, 87.81it/s] 


epoch 0; D_loss: 0.1558; G_loss: 4.9292


100%|██████████| 938/938 [00:10<00:00, 93.77it/s] 


epoch 1; D_loss: 0.1021; G_loss: 4.1357


...

epoch 97; D_loss: 0.7576; G_loss: 1.9347


100%|██████████| 938/938 [00:11<00:00, 84.27it/s]


epoch 98; D_loss: 0.7591; G_loss: 1.9378


100%|██████████| 938/938 [00:11<00:00, 81.99it/s]


epoch 99; D_loss: 0.7563; G_loss: 1.9491
